In [1]:
"""
Tiền xử lý dữ liệu voz_comments.csv cho bài toán phát hiện nội dung vi phạm.
Chạy trên local machine hoặc Google Colab.

Các bước:
  1. Đọc CSV lớn theo chunk (tiết kiệm RAM).
  2. Sửa mojibake còn sót (nếu có) + chuẩn hoá unicode (NFC).
  3. Làm sạch text: bỏ khoảng trắng thừa, HTML entity còn sót, ký tự control.
  4. Lọc: bỏ dòng rỗng / quá ngắn, bỏ trùng lặp theo nội dung text đã chuẩn hoá.
  5. Chia train/val/test theo THREAD (source_url gốc của thread), KHÔNG
     chia ngẫu nhiên theo từng dòng — để tránh rò rỉ dữ liệu: các post
     trong cùng 1 thread có liên quan ngữ cảnh (quote lẫn nhau, cùng chủ
     đề, cùng văn phong của 1 nhóm người tham gia), nếu để lẫn giữa
     train/test sẽ làm điểm đánh giá bị lạc quan giả tạo.
  6. Lưu ra: voz_clean.csv, voz_train.csv, voz_val.csv, voz_test.csv.
"""

import re
import unicodedata
import os
from pathlib import Path

import numpy as np
import pandas as pd

# --------------------------------------------------------------------
# Cấu hình — chạy được trên local machine
# --------------------------------------------------------------------
# Đường dẫn tương đối từ thư mục hiện tại (Data_preprocessing)
IN_PATH = os.path.join("..", "crawler", "data", "voz_comments.csv")
OUT_DIR = os.path.join("..", "crawler", "data")

# Nếu chạy không được, có thể thay đổi thành đường dẫn tuyệt đối:
# IN_PATH = r"e:\ToxicContentDetection\crawler\data\voz_comments.csv"
# OUT_DIR = r"e:\ToxicContentDetection\crawler\data"

MIN_CHARS = 3          # bỏ text ngắn hơn ngưỡng này (vd chỉ 1-2 ký tự, emoji đơn lẻ)
MIN_WORDS = 2           # bỏ text ít hơn số từ này
CHUNK_SIZE = 50_000     # đọc CSV theo chunk để tránh tràn RAM
RANDOM_SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
# phần còn lại (0.1) là test


# %%
# --------------------------------------------------------------------
# 0. Kiểm tra và chuẩn bị thư mục (local machine)
# --------------------------------------------------------------------
def setup_directories():
    """Tạo thư mục output nếu chưa tồn tại, kiểm tra file input."""
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    
    abs_out = os.path.abspath(OUT_DIR)
    abs_in = os.path.abspath(IN_PATH)
    
    print(f"[INFO] Input file:  {abs_in}")
    print(f"[INFO] Output dir:  {abs_out}")
    
    if not os.path.exists(abs_in):
        raise FileNotFoundError(f"File input không tìm thấy: {abs_in}")
    
    print(f"[OK] Tất cả đường dẫn OK!")


# %%
# --------------------------------------------------------------------
# 1. Sửa mojibake còn sót + chuẩn hoá unicode
# --------------------------------------------------------------------
MOJIBAKE_MARKERS = ("Ã", "á»", "áº", "Æ°", "Ä'", "â€", "Â")


def looks_mojibake(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return any(marker in text for marker in MOJIBAKE_MARKERS)


def fix_mojibake(text: str) -> str:
    if not looks_mojibake(text):
        return text
    try:
        fixed = text.encode("cp1252").decode("utf-8")
        return fixed if not looks_mojibake(fixed) else text
    except (UnicodeDecodeError, UnicodeEncodeError):
        return text


def normalize_unicode(text: str) -> str:
    """Chuẩn hoá về dạng NFC — quan trọng với tiếng Việt vì cùng 1 ký tự
    có dấu có thể được biểu diễn bằng nhiều tổ hợp Unicode khác nhau
    (composed vs decomposed), nếu không chuẩn hoá, model coi 2 chuỗi
    'giống hệt nhau khi đọc bằng mắt' là 2 chuỗi khác nhau."""
    return unicodedata.normalize("NFC", text)


# %%
# --------------------------------------------------------------------
# 2. Làm sạch text
# --------------------------------------------------------------------
WHITESPACE_RE = re.compile(r"\s+")
CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
# Vết tích thường gặp khi crawl forum: "Click to expand...", link ảnh dạng
# [IMG]...[/IMG], hoặc BBCode sót lại nếu selector bắt thiếu.
LEFTOVER_ARTIFACTS_RE = re.compile(r"Click to expand\.\.\.$", re.IGNORECASE)


def clean_text(text) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = fix_mojibake(text)
    text = normalize_unicode(text)
    text = CONTROL_CHARS_RE.sub("", text)
    text = LEFTOVER_ARTIFACTS_RE.sub("", text)
    text = WHITESPACE_RE.sub(" ", text)
    return text.strip()


def get_thread_key(row) -> str:
    """Dùng để nhóm các post cùng 1 thread lại với nhau khi chia tập.
    source_url có thể là .../page-2, .../page-3... nên cắt bỏ phần
    '/page-N' để quy về đúng 1 thread gốc."""
    url = row.get("source_url", "") or ""
    return re.sub(r"/page-\d+/?$", "/", url)


# %%
# --------------------------------------------------------------------
# 3. Đọc + làm sạch theo chunk (tránh tràn RAM với file 600MB+)
# --------------------------------------------------------------------
def load_and_clean(in_path: str, chunksize: int = CHUNK_SIZE) -> pd.DataFrame:
    cleaned_chunks = []
    total_raw = 0
    total_kept = 0

    reader = pd.read_csv(
        in_path,
        chunksize=chunksize,
        dtype=str,          # đọc mọi cột dạng string, tự ép kiểu sau nếu cần
        keep_default_na=False,  # tránh pandas tự biến "" hoặc "NA" thành NaN
        encoding="utf-8",
        on_bad_lines='warn',  # bỏ qua dòng bị lỗi (field count mismatch)
        engine='python',  # dùng python engine để xử lý linh hoạt hơn
    )

    for i, chunk in enumerate(reader):
        total_raw += len(chunk)

        chunk["text"] = chunk["text"].map(clean_text)
        if "quoted_text" in chunk.columns:
            chunk["quoted_text"] = chunk["quoted_text"].map(clean_text)
        if "thread_title" in chunk.columns:
            chunk["thread_title"] = chunk["thread_title"].map(clean_text)

        # Lọc text rỗng hoặc quá ngắn
        word_count = chunk["text"].str.split().map(len)
        mask = (chunk["text"].str.len() >= MIN_CHARS) & (word_count >= MIN_WORDS)
        chunk = chunk[mask]

        chunk["thread_key"] = chunk.apply(get_thread_key, axis=1)

        cleaned_chunks.append(chunk)
        total_kept += len(chunk)
        print(f"[CHUNK {i+1}] đọc {len(chunk)} / giữ lại sau lọc: {mask.sum()}")

    df = pd.concat(cleaned_chunks, ignore_index=True)
    print(f"\n[TỔNG] Đọc {total_raw} dòng thô, giữ lại {total_kept} dòng sau lọc rỗng/ngắn.")
    return df


# %%
# --------------------------------------------------------------------
# 4. Loại trùng lặp
# --------------------------------------------------------------------
def dedup(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)

    # Trùng theo nội dung text đã làm sạch
    df = df.drop_duplicates(subset=["text"], keep="first")

    after = len(df)
    print(f"[DEDUP] Bỏ {before - after} dòng trùng nội dung (còn lại {after}).")
    return df


# %%
# --------------------------------------------------------------------
# 5. Chia train/val/test THEO THREAD, không theo từng dòng riên lẻ
# --------------------------------------------------------------------
def split_by_thread(
    df: pd.DataFrame,
    train_ratio: float = TRAIN_RATIO,
    val_ratio: float = VAL_RATIO,
    seed: int = RANDOM_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)

    # Ép về numpy object array trước khi shuffle — cột đọc dạng dtype=str
    # có thể trả về StringArray của pandas, shuffle trực tiếp trên đó
    # không đảm bảo hành vi đúng (pandas cảnh báo), nên ép kiểu tường minh.
    thread_keys = np.array(df["thread_key"].unique(), dtype=object)
    rng.shuffle(thread_keys)

    n = len(thread_keys)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_keys = set(thread_keys[:n_train])
    val_keys = set(thread_keys[n_train:n_train + n_val])
    test_keys = set(thread_keys[n_train + n_val:])

    train_df = df[df["thread_key"].isin(train_keys)].reset_index(drop=True)
    val_df = df[df["thread_key"].isin(val_keys)].reset_index(drop=True)
    test_df = df[df["thread_key"].isin(test_keys)].reset_index(drop=True)

    print(f"[SPLIT] Số thread: train={len(train_keys)} val={len(val_keys)} test={len(test_keys)}")
    print(f"[SPLIT] Số dòng:   train={len(train_df)} val={len(val_df)} test={len(test_df)}")

    return train_df, val_df, test_df


# %%
# --------------------------------------------------------------------
# 6. Chạy toàn bộ pipeline
# --------------------------------------------------------------------
def main():
    print("=" * 70)
    print("BƯỚC 0: Kiểm tra cấu hình")
    print("=" * 70)
    setup_directories()

    print()
    print("=" * 70)
    print("BƯỚC 1: Đọc + làm sạch dữ liệu")
    print("=" * 70)
    df = load_and_clean(IN_PATH)

    print()
    print("=" * 70)
    print("BƯỚC 2: Loại trùng lặp")
    print("=" * 70)
    df = dedup(df)

    print()
    print("=" * 70)
    print("BƯỚC 3: Lưu bản đã làm sạch (toàn bộ, chưa chia tập)")
    print("=" * 70)
    clean_path = os.path.join(OUT_DIR, "voz_clean.csv")
    df.to_csv(clean_path, index=False, encoding="utf-8")
    print(f"[SAVED] {clean_path}  ({len(df)} dòng)")

    print()
    print("=" * 70)
    print("BƯỚC 4: Chia train/val/test theo thread")
    print("=" * 70)
    train_df, val_df, test_df = split_by_thread(df)

    train_df.to_csv(os.path.join(OUT_DIR, "voz_train.csv"), index=False, encoding="utf-8")
    val_df.to_csv(os.path.join(OUT_DIR, "voz_val.csv"), index=False, encoding="utf-8")
    test_df.to_csv(os.path.join(OUT_DIR, "voz_test.csv"), index=False, encoding="utf-8")
    print(f"[SAVED] {OUT_DIR}/voz_train.csv, voz_val.csv, voz_test.csv")

    print()
    print("=" * 70)
    print("THỐNG KÊ NHANH")
    print("=" * 70)
    print(f"Tổng dòng sau làm sạch : {len(df)}")
    print(f"Số thread duy nhất     : {df['thread_key'].nunique()}")
    print(f"Độ dài text trung bình : {df['text'].str.len().mean():.1f} ký tự")
    print(f"Độ dài text median     : {df['text'].str.len().median():.0f} ký tự")


if __name__ == "__main__":
    main()


BƯỚC 0: Kiểm tra cấu hình
[INFO] Input file:  e:\ToxicContentDetection\crawler\data\voz_comments.csv
[INFO] Output dir:  e:\ToxicContentDetection\crawler\data
[OK] Tất cả đường dẫn OK!

BƯỚC 1: Đọc + làm sạch dữ liệu
[CHUNK 1] đọc 49847 / giữ lại sau lọc: 49847
[CHUNK 2] đọc 49691 / giữ lại sau lọc: 49691
[CHUNK 3] đọc 49673 / giữ lại sau lọc: 49673
[CHUNK 4] đọc 49786 / giữ lại sau lọc: 49786
[CHUNK 5] đọc 49800 / giữ lại sau lọc: 49800
[CHUNK 6] đọc 49883 / giữ lại sau lọc: 49883
[CHUNK 7] đọc 49865 / giữ lại sau lọc: 49865
[CHUNK 8] đọc 49833 / giữ lại sau lọc: 49833
[CHUNK 9] đọc 49801 / giữ lại sau lọc: 49801
[CHUNK 10] đọc 49740 / giữ lại sau lọc: 49740
[CHUNK 11] đọc 49693 / giữ lại sau lọc: 49693
[CHUNK 12] đọc 49533 / giữ lại sau lọc: 49533
[CHUNK 13] đọc 49587 / giữ lại sau lọc: 49587
[CHUNK 14] đọc 49575 / giữ lại sau lọc: 49575
[CHUNK 15] đọc 49595 / giữ lại sau lọc: 49595
[CHUNK 16] đọc 49463 / giữ lại sau lọc: 49463
[CHUNK 17] đọc 49319 / giữ lại sau lọc: 49319
[CHUNK 18]

C:\Users\PC\AppData\Local\Temp\ipykernel_9364\2291538626.py:146: ParserWarning: Skipping line 1498690: Expected 11 fields in line 1498690, saw 21

  for i, chunk in enumerate(reader):


[CHUNK 30] đọc 49820 / giữ lại sau lọc: 49820
[CHUNK 31] đọc 49806 / giữ lại sau lọc: 49806
[CHUNK 32] đọc 49630 / giữ lại sau lọc: 49630
[CHUNK 33] đọc 49871 / giữ lại sau lọc: 49871


C:\Users\PC\AppData\Local\Temp\ipykernel_9364\2291538626.py:146: ParserWarning: Skipping line 1656979: Expected 11 fields in line 1656979, saw 21

  for i, chunk in enumerate(reader):


[CHUNK 34] đọc 49805 / giữ lại sau lọc: 49805
[CHUNK 35] đọc 49725 / giữ lại sau lọc: 49725
[CHUNK 36] đọc 49758 / giữ lại sau lọc: 49758


...

[CHUNK 72] đọc 49885 / giữ lại sau lọc: 49885
[CHUNK 73] đọc 49925 / giữ lại sau lọc: 49925
[CHUNK 74] đọc 49889 / giữ lại sau lọc: 49889
[CHUNK 75] đọc 49887 / giữ lại sau lọc: 49887
[CHUNK 76] đọc 49873 / giữ lại sau lọc: 49873
[CHUNK 77] đọc 49906 / giữ lại sau lọc: 49906
[CHUNK 78] đọc 49907 / giữ lại sau lọc: 49907
[CHUNK 79] đọc 49733 / giữ lại sau lọc: 49733
[CHUNK 80] đọc 49746 / giữ lại sau lọc: 49746


C:\Users\PC\AppData\Local\Temp\ipykernel_9364\2291538626.py:146: ParserWarning: Skipping line 4039307: Expected 11 fields in line 4039307, saw 21

  for i, chunk in enumerate(reader):


[CHUNK 81] đọc 49854 / giữ lại sau lọc: 49854
[CHUNK 82] đọc 49852 / giữ lại sau lọc: 49852
[CHUNK 83] đọc 49830 / giữ lại sau lọc: 49830
[CHUNK 84] đọc 49834 / giữ lại sau lọc: 49834
[CHUNK 85] đọc 49862 / giữ lại sau lọc: 49862
[CHUNK 86] đọc 49799 / giữ lại sau lọc: 49799
[CHUNK 87] đọc 49729 / giữ lại sau lọc: 49729
[CHUNK 88] đọc 49840 / giữ lại sau lọc: 49840
[CHUNK 89] đọc 49798 / giữ lại sau lọc: 49798
[CHUNK 90] đọc 49643 / giữ lại sau lọc: 49643
[CHUNK 91] đọc 49741 / giữ lại sau lọc: 49741
[CHUNK 92] đọc 49258 / giữ lại sau lọc: 49258
[CHUNK 93] đọc 47965 / giữ lại sau lọc: 47965


C:\Users\PC\AppData\Local\Temp\ipykernel_9364\2291538626.py:146: ParserWarning: Skipping line 4694995: Expected 11 fields in line 4694995, saw 21

  for i, chunk in enumerate(reader):


[CHUNK 94] đọc 47684 / giữ lại sau lọc: 47684
[CHUNK 95] đọc 48470 / giữ lại sau lọc: 48470
[CHUNK 96] đọc 49871 / giữ lại sau lọc: 49871
[CHUNK 97] đọc 49876 / giữ lại sau lọc: 49876
[CHUNK 98] đọc 49879 / giữ lại sau lọc: 49879
[CHUNK 99] đọc 49884 / giữ lại sau lọc: 49884
[CHUNK 100] đọc 48404 / giữ lại sau lọc: 48404
[CHUNK 101] đọc 49237 / giữ lại sau lọc: 49237
[CHUNK 102] đọc 49706 / giữ lại sau lọc: 49706
[CHUNK 103] đọc 49733 / giữ lại sau lọc: 49733
[CHUNK 104] đọc 49859 / giữ lại sau lọc: 49859
[CHUNK 105] đọc 49844 / giữ lại sau lọc: 49844
[CHUNK 106] đọc 49850 / giữ lại sau lọc: 49850
[CHUNK 107] đọc 49802 / giữ lại sau lọc: 49802
[CHUNK 108] đọc 49866 / giữ lại sau lọc: 49866
[CHUNK 109] đọc 49879 / giữ lại sau lọc: 49879
[CHUNK 110] đọc 49907 / giữ lại sau lọc: 49907
[CHUNK 111] đọc 49827 / giữ lại sau lọc: 49827
[CHUNK 112] đọc 49635 / giữ lại sau lọc: 49635
[CHUNK 113] đọc 49575 / giữ lại sau lọc: 49575
[CHUNK 114] đọc 49117 / giữ lại sau lọc: 49117
[CHUNK 115] đọc 234